# Mnemo LIVE DEMO: any first-person video → diary → your questions

**Settings (right panel):** Accelerator **GPU T4** · Internet **On** · Input: dataset **mnemofyp**.

**Before the guide arrives:** run steps 1 and 2 (about 8 minutes). After that everything is loaded and waiting.

**In front of the guide:**
3. Give it a video (upload or link), then run step 3
4. Run step 4: Mnemo watches the video and writes the diary (about 1.5× the video length, so keep it to 2-3 minutes)
5. Run step 5 and type questions. Press Enter on an empty box to stop.

New video? Just run steps 3, 4, 5 again. Nothing has to be reloaded.

## 1. Code + packages (~3 min)

In [ ]:
import os, shutil, glob, sys, subprocess, json, time
WORK = '/kaggle/working'
CODE = f'{WORK}/mnemo'
code_dir = code_zip = None
for root, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'mnemo.py' in files and 'modules' in dirs:
        code_dir = root
    if 'mnemo_code.zip' in files:
        code_zip = os.path.join(root, 'mnemo_code.zip')
print('code folder:', code_dir, '| zip:', code_zip)
assert code_dir or code_zip, 'Code not found: add the mnemofyp dataset (Add Input) and wait until it is attached'
shutil.rmtree(CODE, ignore_errors=True)
if code_dir:
    shutil.copytree(code_dir, CODE)
else:
    os.makedirs(CODE)
    !cd "{CODE}" && unzip -q "{code_zip}"
%cd {CODE}
!pip install -q -U "transformers>=4.57" accelerate mediapipe ultralytics gdown
import torch, transformers
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE: switch Accelerator to GPU T4!')

## 2. Load the models and warm them up (~5 min)

In [ ]:
import numpy as np
import config as C
from modules.vlm_logger import VLMDescriber, ActionLogger
from modules.hoi_gate import HOIGate, gate_decisions
t = time.time()
describer = VLMDescriber(C.VLM_MODEL)            # watches the video (stage 1) and answers (stage 2)
gate = HOIGate('models/hoi_tcn.pt')               # our trigger: hands + objects + HOI-TCN
describer.describe([np.zeros((360, 640, 3), np.uint8)] * C.VLM_FRAMES, C.VLM_WINDOW_S)   # warm-up: the first call is slow
describer.chat('Say OK.')
print(f'READY in {time.time() - t:.0f}s | GPU memory {torch.cuda.memory_allocated() / 1e9:.1f} GB')

## 3. Give it a video
Pick **one** way:
- **A. Upload:** right panel → **Add Input** → **Upload** → drop the video file → give it any name → **Create**.
  Wait until it shows under Input, then run this cell. (The newest video under Input is used.)
- **B. Link:** a Google Drive link (sharing: *anyone with the link*) or any direct video link → paste it in `VIDEO_LINK`.

Any phone video works (mp4 / mov, portrait or landscape). It is converted to a small 640 px copy here.

In [ ]:
VIDEO_LINK = ''        # B: paste a link here, or leave '' to use the newest uploaded video (A)
MAX_MINUTES = 5        # longer videos are cut (processing takes about 1.5x the video length)

EXT = ('.mp4', '.mov', '.m4v', '.avi', '.mkv', '.webm', '.3gp')
if VIDEO_LINK:
    import gdown
    RAW = gdown.download(VIDEO_LINK, f'{WORK}/input_video_raw', fuzzy=True, quiet=True)
else:
    found = [os.path.join(r, f) for r, _, fs in os.walk('/kaggle/input', followlinks=True)
             for f in fs if f.lower().endswith(EXT)]
    assert found, 'No video found under Input: upload one (A) or paste a link (B)'
    RAW = max(found, key=os.path.getmtime)
    print('videos under Input:', *[os.path.relpath(f, '/kaggle/input') for f in found], sep='\n  ')
print('using:', RAW)

VIDEO = f'{WORK}/live.mp4'
!ffmpeg -y -loglevel error -i "{RAW}" -t {MAX_MINUTES * 60} -vf "scale=640:-2,fps=30" -c:v libx264 -preset veryfast -crf 23 -an "{VIDEO}"
import cv2
cap = cv2.VideoCapture(VIDEO)
DURATION = cap.get(cv2.CAP_PROP_FRAME_COUNT) / (cap.get(cv2.CAP_PROP_FPS) or 30)
cap.release()
print(f'video ready: {DURATION:.0f} s → expect about {DURATION * 1.6 / 60:.0f}-{DURATION * 2 / 60 + 1:.0f} min of processing')

import matplotlib.pyplot as plt
from evaluation.qa_benchmark import clip_frames
thumbs, _ = clip_frames(VIDEO, 6)
fig, ax = plt.subplots(1, len(thumbs), figsize=(16, 3))
for a, f in zip(np.atleast_1d(ax), thumbs):
    a.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)); a.axis('off')
plt.show()

## 4. Mnemo watches the video and writes the diary
- `USE_TRIGGER = False`: the model looks at **every** 4-second window (safest on a new place, finds the most).
- `USE_TRIGGER = True`: **our trigger** skips windows with no hand interaction (faster; shows `skipped`).
  It was trained on kitchen video, so outside a kitchen it may skip a few real actions.

Diary times are **real clock times**: the video is treated as if it just ended now.

In [ ]:
from datetime import datetime, timedelta
from zoneinfo import ZoneInfo
from modules.storage import EventStore
from query.llm_qa import fmt_row

USE_TRIGGER = False
KEEP_PREVIOUS = False          # True = keep the diary of earlier videos too (ask about all of them)
TZ = ZoneInfo('Asia/Kolkata')  # Kaggle's clock is UTC; this makes the diary show your local time
def now_local():
    return datetime.now(TZ).replace(tzinfo=None)

store = EventStore(f'{WORK}/mnemo_live.db')
if not KEEP_PREVIOUS:
    store.clear_activities()
START = now_local() - timedelta(seconds=DURATION)

call = scores = None
if USE_TRIGGER:
    sig, g = gate.signals(VIDEO)
    scores = sig['hoi']
    call = gate_decisions(scores)
    print(f'TRIGGER: model needed on {call.sum()} of {len(call)} windows (trigger took {g["gate_seconds"]}s)\n')
logger = ActionLogger(describer, store)
rows, stats = logger.run(VIDEO, start_time=START, call=call, scores=scores)
print(f"\n{stats['windows']} windows, model called on {stats['vlm_calls']}, "
      f"{stats['wall_seconds'] / 60:.1f} min for {stats['video_seconds']:.0f} s of video")

print('\n══ THE DIARY (this text is ALL that is kept; the video is not stored) ══')
for r in store.activities():
    if r['verb']:
        print('  ', fmt_row(r))
for obj, (st, ts) in logger.tracker.current().items():
    if st in ('open', 'on'):
        print(f'\n⚠  ALERT: the {obj.removesuffix(" power")} was last seen {st.upper()} at {ts:%I:%M:%S %p}')

## 5. Ask questions
Type a question and press Enter. **Empty question = stop.** Answers come ONLY from the diary above,
with the time, or *"I didn't see that in my records."*

Tips: ask about actions (*Did I open the cupboard? When did I drink water? What did I do?*).
Also ask about something that did **not** happen (*Did I take my medicine?*): Mnemo should not invent it.
Say *today* or no time word; *this morning* only covers until 12:00.

In [ ]:
from query.llm_qa import answer, retrieve
SHOW_EVIDENCE = True     # also print the diary lines that match the question
while True:
    q = input('\nQuestion (Enter on empty = stop): ').strip()
    if not q:
        break
    r = answer(q, store, llm=describer, now=now_local())
    print('MNEMO:', r['answer'])
    if SHOW_EVIDENCE:
        hits = retrieve([x for x in store.activities() if x['verb']], q, k=4)
        for x in sorted(hits, key=lambda x: x['ts_start']):
            print('   diary:', fmt_row(x))

## 6. (Optional) Same question to the model directly, with no diary
The big model gets 32 frames of the whole video and the question. On a short video this is strong too.
The diary wins on **long** periods (a whole morning) and on **"when"** questions, because it keeps times.

In [ ]:
from evaluation.qa_benchmark import clip_frames
direct_frames, direct_sec = clip_frames(VIDEO, 32)
while True:
    q = input('\nQuestion for the model directly (Enter on empty = stop): ').strip()
    if not q:
        break
    print('DIRECT MODEL:', describer.ask_video(direct_frames, direct_sec, q))